# Explainable AI Backend — original 13 nodes kept as-is + 2 new conditional nodes

**Original nodes, unchanged:** query_analysis, missing_context_detector, reasoning_evidence_planner,
source_selection, tool_execution, evidence_aggregation, candidate_generation, evaluation, select_best,
final_answer, explainability, reasoning_tree, summary.

**New nodes added:**
- `rag_node` — runs only when the frontend sends `document_uploaded=True` (user hit enter on an uploaded
  document). Retrieves from that document's vector store and feeds the result into evidence_aggregation
  alongside tool_results.
- `research_node` — runs only when `reasoning_evidence_planner` classified the query as `"Complex"`.
  Searches for relevant research papers (arXiv, PubMed, Google Scholar via DuckDuckGo site-scoping),
  summarizes findings, and feeds them into candidate_generation alongside the aggregated evidence.

**New routing behavior:**
- `Simple` queries skip `tool_execution` entirely — no multi-tool search for "what's 2+2" or a greeting.
- `Moderate`/`Complex` queries still run the full multi-tool sweep exactly as before.
- Only `Complex` queries additionally run `research_node`.
- `rag_node` runs independently of complexity — it's driven purely by whether a document was uploaded this turn.


In [ ]:
from __future__ import annotations
import os
import json
import re
from typing import TypedDict, List, Dict, Any, Optional

import requests
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_core.tools import tool
from langchain_community.tools import DuckDuckGoSearchResults
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings

load_dotenv()

llm = ChatOpenAI(
    model="gpt-5-mini",
    api_key=os.getenv("OPENAI_API_KEY"),
)

embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2-preview")

NEWSDATA_API_KEY = os.getenv("NEWSDATA_API_KEY")
ALPHA_VANTAGE_API_KEY = os.getenv("ALPHA_VANTAGE_API_KEY")
OPENWEATHER_API_KEY = os.getenv("OPENWEATHER_API_KEY")

for name, val in [
    ("OPENAI_API_KEY", os.getenv("OPENAI_API_KEY")),
    ("NEWSDATA_API_KEY", NEWSDATA_API_KEY),
    ("ALPHA_VANTAGE_API_KEY", ALPHA_VANTAGE_API_KEY),
    ("OPENWEATHER_API_KEY", OPENWEATHER_API_KEY),
]:
    if not val:
        print(f"WARNING: {name} is not set in your environment/.env file.")


## JSON parsing helper — unchanged from your original

In [ ]:
def _extract_json_block(text):
    text = text.strip()
    fence_match = re.search(r"```(?:json)?\s*(\{.*\}|\[.*\])\s*```", text, re.DOTALL)
    if fence_match:
        return fence_match.group(1).strip()
    for open_ch, close_ch in (("{", "}"), ("[", "]")):
        start = text.find(open_ch)
        if start == -1:
            continue
        depth = 0
        for i in range(start, len(text)):
            if text[i] == open_ch:
                depth += 1
            elif text[i] == close_ch:
                depth -= 1
                if depth == 0:
                    return text[start:i + 1]
    return text


def parse_json_response(response):
    content = response.content
    if isinstance(content, str):
        text = content
    elif isinstance(content, list):
        text = "".join(item.get("text", "") for item in content if isinstance(item, dict))
    else:
        raise ValueError(f"Unsupported response type: {type(content)}")

    candidate = _extract_json_block(text)
    try:
        return json.loads(candidate)
    except json.JSONDecodeError:
        print("Raw LLM Output:")
        print(text)
        print("\n--- Extracted candidate JSON ---")
        print(candidate)
        raise ValueError(
            "LLM did not return valid JSON. This is often caused by the "
            "response being cut off - if the output above looks truncated "
            "(missing a final closing brace), raise max_output_tokens on "
            "the llm, or simplify the prompt's requested output."
        )


## State

Original fields kept, plus new ones for the document/RAG flow and research node: `document_uploaded`, `document_path`, `pdf_context`, `research_findings`.

In [ ]:
class State(TypedDict):

    user_query: str
    original_query: str

    # --- new: document / RAG flow ---
    document_uploaded: bool
    document_path: Optional[str]
    pdf_context: List[str]

    # Node 1
    query_analysis: Dict

    # Node 1.5 - Missing Context Detector
    missing_context: Dict

    # Node 1.75 - Reasoning & Evidence Planner
    reasoning_plan: Dict

    # Node 2
    source_selection: Dict

    # Node 2.5 - Tool Execution
    tool_results: List[Dict]

    # --- new: Research Agent (complex queries only) ---
    research_findings: Dict

    # Node 2.75 - Evidence Aggregation
    evidence: Dict

    # Node 3
    candidate_approaches: List[Dict]

    # Node 4
    evaluations: List[Dict]

    # Node 5
    selected_approach: Dict
    rejected_approaches: List[Dict]

    # Node 6
    answer: str

    # Node 7
    explainability: Dict

    # Node 8
    reasoning_tree: Dict

    # Node 9
    summary: Dict


## Original Node 1 — `query_analysis` (unchanged)

In [ ]:
def query_analysis(state: State) -> State:
    query = state["user_query"]

    system_prompt = """
You are the Query Analysis Engine for an Explainable AI system.

Analyze the user's query.

Return ONLY valid JSON.

{
    "intent": "",
    "domain": "",
    "query_type": "",
    "complexity": "",
    "ambiguity": {
        "is_ambiguous": false,
        "reason": ""
    },
    "entities": [],
    "keywords": [],
    "required_knowledge": [],
    "possible_user_goal": "",
    "constraints": [],
    "reasoning_type": "",
    "risk_level": "",
    "confidence": 0,
    "analysis_summary": ""
}

Rules:
- Do NOT answer the user's question.
- Only analyze the query.
- Return ONLY valid JSON.
"""

    response = llm.invoke([
        SystemMessage(content=system_prompt),
        HumanMessage(content=query)
    ])

    analysis = parse_json_response(response)

    state["original_query"] = query
    state["query_analysis"] = analysis

    return state


## Original Node 1.5 — `missing_context_detector` (unchanged)

In [ ]:
MISSING_CONTEXT_PROMPT = """
You are the Missing Context Detector for an Explainable AI system.

You are given the user's query and its Query Analysis.

Your job is to decide whether the system needs to ask the user follow-up
questions before it can answer safely and accurately.

Use the "ambiguity" field from the Query Analysis as your primary signal,
but also use your own judgement (e.g. missing patient details for a medical
question, missing location for a weather question, missing symbol for a
stock question, missing timeframe for a news question).

Rules:
- Ask AT MOST 4 questions.
- Only ask questions that materially change the answer.
- Questions must be short, specific, and directly answerable by the user.
- Do NOT answer the user's original query.
- Return ONLY valid JSON.

Return exactly this format:

{
    "needs_clarification": false,
    "questions": []
}
"""

def missing_context_detector(state: State) -> State:

    human_prompt = f"""
User Query:
{state["user_query"]}

Query Analysis:
{json.dumps(state["query_analysis"], indent=2)}
"""

    response = llm.invoke([
        SystemMessage(content=MISSING_CONTEXT_PROMPT),
        HumanMessage(content=human_prompt)
    ])

    result = parse_json_response(response)

    needs_clarification = bool(result.get("needs_clarification", False))
    questions = result.get("questions", []) or []

    answers = []

    if needs_clarification and questions:
        print("\nThe system needs a few more details before it can answer accurately:\n")
        for q in questions:
            ans = input(f"{q}\n> ")
            answers.append({"question": q, "answer": ans})

    if answers:
        lines = []
        for a in answers:
            lines.append("- " + a["question"] + " -> " + a["answer"])
        extra_context = "\n".join(lines)

        clarified_query = (
            state["user_query"]
            + "\n\nAdditional details provided by the user:\n"
            + extra_context
        )
    else:
        clarified_query = state["user_query"]

    state["missing_context"] = {
        "needs_clarification": needs_clarification,
        "questions": questions,
        "answers": answers
    }

    state["user_query"] = clarified_query

    return state


## Original Node 1.75 — `reasoning_evidence_planner` (unchanged)

This is the node whose `query_complexity` output now actually drives routing (see the graph section below) instead of being descriptive-only.

In [ ]:
REASONING_EVIDENCE_PLANNER_PROMPT = """
You are the Reasoning & Evidence Planner of an Explainable AI system.
Your responsibility is to determine the most effective reasoning strategy before generating any answer.

Inputs:
1. User Query
2. Query Analysis
3. Missing Context Result

Available Tools:
- LLM (Reasoning and knowledge synthesis)
- DuckDuckGo Search (General web search)
- News API (Latest news and current events)
- Stock Market API (Financial and market data)

Tasks:
1. Analyze the query complexity and classify it as:
   - Simple
   - Moderate
   - Complex
2. Determine the reasoning strategy:
   - Simple Reasoning
   - Multi-Step Reasoning
   - Multi-Source Reasoning
3. Select ONLY the tools that add meaningful value.
   - Do not select unnecessary tools.
   - A tool should only be selected if it improves the quality, accuracy, freshness, or reliability of the answer.
   - If internal reasoning is sufficient, use only the LLM.
   - If current or factual information is required, include the appropriate external tools.
4. For every selected tool provide:
   - tool_name
   - priority (0-100)
   - expected_contribution (0-100)
   - expected_trust (0-100)
   - reason_for_selection
5. For every unselected tool provide:
   - tool_name
   - reason_not_selected
6. Decide:
   - minimum number of evidence sources required
   - number of candidate approaches to generate
7. Estimate:
   - expected overall confidence (0-100)

Rules:
- Use the minimum number of tools required.
- Prefer recent and reliable information for dynamic queries.
- For finance-related queries, prioritize the Stock Market API.
- For current events, prioritize the News API.
- For general web information, use DuckDuckGo Search.
- If multiple tools provide complementary information, combine them.
- Do not generate the final answer.
- Return ONLY valid JSON.

JSON Format:
{
  "query_complexity": "",
  "reasoning_strategy": "",
  "selected_tools": [
    {
      "tool_name": "",
      "priority": 95,
      "expected_contribution": 40,
      "expected_trust": 98,
      "reason_for_selection": ""
    }
  ],
  "ignored_tools": [
    {
      "tool_name": "",
      "reason_not_selected": ""
    }
  ],
  "minimum_evidence_sources": 1,
  "candidate_approaches": 5,
  "expected_confidence": 95,
  "planning_summary": ""
}
"""

def reasoning_evidence_planner(state: State) -> State:

    human_prompt = f"""
User Query:
{state["user_query"]}

Query Analysis:
{json.dumps(state["query_analysis"], indent=2)}

Missing Context Result:
{json.dumps(state["missing_context"], indent=2)}
"""

    response = llm.invoke([
        SystemMessage(content=REASONING_EVIDENCE_PLANNER_PROMPT),
        HumanMessage(content=human_prompt)
    ])

    result = parse_json_response(response)

    state["reasoning_plan"] = result

    return state


## Original Node 2 — `source_selection` (unchanged)

In [ ]:
SOURCE_SELECTION_PROMPT = """
You are the Source Selection Engine for an Explainable AI system.

Your task is to determine whether answering the user's query requires external information.

Available Sources:

1. LLM Internal Knowledge
2. Google Search
3. Wikipedia
4. Research Papers
5. Government Websites
6. Official Documentation
7. News Articles
8. Books
9. Stack Overflow
10. GitHub

Instructions:

Analyze the query and decide:

- Does the query require external sources?
- Which sources should be used?
- Why should each source be used?
- Assign a priority (1 = highest).
- Assign a confidence score (0-100).

Do NOT answer the user's question.

Return ONLY valid JSON.

{
  "requires_external_sources": true,
  "selected_sources": [
    {
      "source": "",
      "priority": 1,
      "confidence": 95,
      "reason": ""
    }
  ],
  "source_summary": ""
}
"""

def source_selection(state: State) -> State:

    human_prompt = f"""
User Query:
{state["user_query"]}

Query Analysis:
{json.dumps(state["query_analysis"], indent=2)}
"""

    response = llm.invoke([
        SystemMessage(content=SOURCE_SELECTION_PROMPT),
        HumanMessage(content=human_prompt)
    ])

    result = parse_json_response(response)

    state["source_selection"] = result

    return state


## NEW Node — `rag_node`

Fires only when `document_uploaded=True` in the input state (your frontend sets this when the user hits enter on an uploaded document). Builds/reuses a FAISS index for `document_path` and retrieves the top chunks for the query. Runs independently of query complexity — a document was explicitly provided, so it's always consulted regardless of how simple the query looks.

In [ ]:
_vector_store_cache: Dict[str, Any] = {}


def _get_retriever_for_document(document_path: str):
    if document_path not in _vector_store_cache:
        loader = PyPDFLoader(document_path)
        docs = loader.load()
        chunks = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200).split_documents(docs)
        vector_store = FAISS.from_documents(chunks, embeddings)
        _vector_store_cache[document_path] = vector_store
    return _vector_store_cache[document_path].as_retriever(search_type="similarity", search_kwargs={"k": 4})


def rag_node(state: State) -> State:
    document_path = state.get("document_path")
    if not document_path:
        state["pdf_context"] = []
        return state

    retriever = _get_retriever_for_document(document_path)
    results = retriever.invoke(state["user_query"])
    state["pdf_context"] = [doc.page_content for doc in results]
    return state


## Original Node 2.5 — `tool_execution` (unchanged)

Still only called when the graph routes into it — see routing rule below: skipped entirely for `Simple` queries.

In [ ]:
from langchain_community.tools import DuckDuckGoSearchResults

search_tool = DuckDuckGoSearchResults(region="us-en")


@tool
def calculator(num1: float, num2: float, op: str) -> dict:
    """
    Perform basic arithmetic operations on two numbers.
    Supported operations: add, subtract, multiply, divide
    """
    try:
        if op == "add":
            result = num1 + num2
        elif op == "subtract":
            result = num1 - num2
        elif op == "multiply":
            result = num1 * num2
        elif op == "divide":
            if num2 == 0:
                return {"error": "Cannot divide by zero"}
            result = num1 / num2
        else:
            return {"error": f"Unsupported operation '{op}'. Use add, subtract, multiply, or divide."}
        return {"num1": num1, "num2": num2, "operation": op, "result": result}
    except Exception as e:
        return {"error": str(e)}


@tool
def get_stock_price(symbol: str) -> dict:
    """
    Fetch latest stock price for a given symbol (e.g. 'AAPL', 'TSLA')
    using the Alpha Vantage API.
    """
    url = (
        "https://www.alphavantage.co/query"
        f"?function=GLOBAL_QUOTE&symbol={symbol}&apikey={ALPHA_VANTAGE_API_KEY}"
    )
    r = requests.get(url, timeout=15)
    return r.json()


@tool
def get_weather(city: str) -> dict:
    """
    Fetch current weather for a given city using the OpenWeatherMap API.
    """
    url = (
        "https://api.openweathermap.org/data/2.5/weather"
        f"?q={city}&appid={OPENWEATHER_API_KEY}&units=metric"
    )
    r = requests.get(url, timeout=15)
    return r.json()


@tool
def get_news(query: str) -> dict:
    """
    Fetch the latest news articles related to a query using the NewsData.io API.
    Returns article titles, links, and sources - use these links when citing
    news-based claims in the final answer.
    """
    url = "https://newsdata.io/api/1/latest"
    params = {"apikey": NEWSDATA_API_KEY, "q": query}
    r = requests.get(url, params=params, timeout=15)
    return r.json()


available_tools = [search_tool, calculator, get_stock_price, get_weather, get_news]
llm_with_tools = llm.bind_tools(available_tools)

PLANNER_TOOL_MAP = {
    "DuckDuckGo Search": search_tool,
    "News API": get_news,
    "Stock Market API": get_stock_price,
}

TOOL_ROUTING_PROMPT = """
You are the Tool Execution Router for an Explainable AI system.

The Reasoning & Evidence Planner has already decided WHICH tools are worth
calling for this query. Your only job is to call those tools with the right
arguments to actually fetch the evidence.

Only call tools from the ones made available to you in this turn. Call each
relevant tool at most once, with a well-formed query/argument based on the
user's query. If none of the available tools would return anything useful
for this specific query, call no tools at all.
"""

def tool_execution(state: State) -> State:

    plan = state.get("reasoning_plan", {}) or {}
    selected = plan.get("selected_tools", []) or []

    planned_tools = []
    for entry in selected:
        name = entry.get("tool_name", "")
        mapped = PLANNER_TOOL_MAP.get(name)
        if mapped is not None and mapped not in planned_tools:
            planned_tools.append(mapped)

    if not planned_tools:
        planned_tools = available_tools

    scoped_llm_with_tools = llm.bind_tools(planned_tools)

    human_prompt = f"""
User Query:
{state["user_query"]}

Query Analysis:
{json.dumps(state["query_analysis"], indent=2)}

Reasoning & Evidence Plan:
{json.dumps(plan, indent=2)}
"""

    response = scoped_llm_with_tools.invoke([
        SystemMessage(content=TOOL_ROUTING_PROMPT),
        HumanMessage(content=human_prompt)
    ])

    tool_results = []

    for call in getattr(response, "tool_calls", None) or []:
        name = call["name"]
        args = call.get("args", {})

        matching_tool = next(
            (t for t in planned_tools if t.name == name), None
        )

        if matching_tool is None:
            tool_results.append({
                "tool": name,
                "input": args,
                "output": {"error": f"Unknown tool '{name}'"}
            })
            continue

        try:
            output = matching_tool.invoke(args)
        except Exception as e:
            output = {"error": str(e)}

        tool_results.append({
            "tool": name,
            "input": args,
            "output": output
        })

    state["tool_results"] = tool_results

    return state


## NEW Node — `research_node`

Only runs for `Complex` queries (checked in the routing function below). Searches DuckDuckGo scoped to research sources (arXiv, PubMed, Google Scholar, IEEE, ACM, Springer, ScienceDirect, Nature), keeps it to ~5 results, and summarizes — matching the "quality over quantity" rule from your original research-agent spec.

In [ ]:
RESEARCH_SOURCE_SITES = [
    "arxiv.org", "pubmed.ncbi.nlm.nih.gov", "scholar.google.com",
    "ieeexplore.ieee.org", "dl.acm.org", "link.springer.com", "sciencedirect.com", "nature.com",
]

RESEARCH_SUMMARY_PROMPT = """
You are the Research Agent of an Explainable AI system.

You are given the user's query and raw search results scoped to trusted
research sources (arXiv, PubMed, Google Scholar, IEEE, ACM, Springer,
ScienceDirect, Nature).

Your task:
- Identify the core topic.
- Extract key findings from up to 5 of the most relevant results.
- Note agreements and contradictions between them, if any.
- Produce a short structured summary of the evidence.

Rules:
- Do NOT fabricate papers or findings not present in the raw results.
- Do NOT answer the user's query.
- Return ONLY valid JSON.

Format:
{
    "core_topic": "",
    "key_findings": [
        {"title": "", "summary": "", "url": ""}
    ],
    "agreements": [],
    "contradictions": [],
    "evidence_summary": ""
}
"""

def research_node(state: State) -> State:
    query = state["user_query"]
    scoped_query = query + " " + " OR ".join(f"site:{s}" for s in RESEARCH_SOURCE_SITES)

    try:
        raw_results = search_tool.invoke(scoped_query)
    except Exception as e:
        raw_results = {"error": str(e)}

    human_prompt = f"""
User Query:
{query}

Raw Research Search Results:
{raw_results}
"""

    response = llm.invoke([
        SystemMessage(content=RESEARCH_SUMMARY_PROMPT),
        HumanMessage(content=human_prompt)
    ])

    state["research_findings"] = parse_json_response(response)
    return state


## Original Node 2.75 — `evidence_aggregation` (unchanged logic, extra input)

Same prompt and behavior as before. It now also receives `pdf_context` from `rag_node` alongside `tool_results`, so document evidence and tool evidence get aggregated together the same way tool evidence alone used to be.

In [ ]:
EVIDENCE_AGGREGATION_PROMPT = """
You are the Evidence Aggregation Engine for an Explainable AI system.

You are given:
1. User Query
2. Query Analysis
3. Reasoning & Evidence Plan
4. Raw Tool Results (outputs from the tools that were actually executed)
5. Document Context (chunks retrieved from an uploaded document, if any)

Your task is to turn the raw tool outputs and document context into a clean,
structured evidence list that downstream reasoning nodes can rely on.

For each piece of evidence, extract:
- source (tool name, e.g. "DuckDuckGo Search", "News API", "Stock Market API", "Uploaded Document")
- title
- summary (1-2 sentences, in your own words)
- url (exact link if the raw tool result contains one, otherwise empty string)
- trust_score (0-100, your judgement of how reliable this specific piece is)
- relevance_score (0-100, how relevant this is to the user's query)

Rules:
- Only include evidence that is genuinely useful. Drop noise and duplicates.
- If a tool call returned an error, do NOT include it as evidence - note it
  in "tool_errors" instead.
- Do NOT fabricate URLs or facts that are not present in the raw tool results.
- Do NOT answer the user's query.
- Return ONLY valid JSON.

Format:

{
  "evidence": [
    {
      "source": "",
      "title": "",
      "summary": "",
      "url": "",
      "trust_score": 0,
      "relevance_score": 0
    }
  ],
  "tool_errors": [
    {
      "tool": "",
      "error": ""
    }
  ],
  "evidence_summary": ""
}
"""

def evidence_aggregation(state: State) -> State:

    human_prompt = f"""
User Query:
{state["user_query"]}

Query Analysis:
{json.dumps(state["query_analysis"], indent=2)}

Reasoning & Evidence Plan:
{json.dumps(state.get("reasoning_plan", {}), indent=2)}

Raw Tool Results:
{json.dumps(state.get("tool_results", []), indent=2, default=str)}

Document Context:
{json.dumps(state.get("pdf_context", []), indent=2, default=str)}
"""

    response = llm.invoke([
        SystemMessage(content=EVIDENCE_AGGREGATION_PROMPT),
        HumanMessage(content=human_prompt)
    ])

    result = parse_json_response(response)

    state["evidence"] = result

    return state


## Original Nodes 3-9 — unchanged

`generate_candidate_approaches`, `evaluate_candidate_approaches`, `select_best_approach`, `generate_final_answer`, `generate_explainability`, `generate_summary`, `generate_reasoning_tree` — identical to your original file. `generate_candidate_approaches` and `generate_final_answer` now also see `research_findings` when it ran.

In [ ]:
def generate_candidate_approaches(state: State) -> State:

    plan = state.get("reasoning_plan", {}) or {}
    approach_count = plan.get("candidate_approaches", 5)
    if not isinstance(approach_count, int) or approach_count < 1:
        approach_count = 5

    prompt = f"""
                    You are the Candidate Strategy Generator for an Explainable AI system.

                    You are given:
                    1. User Query
                    2. Query Analysis
                    3. Source Selection
                    4. Reasoning & Evidence Plan (recommended reasoning strategy)
                    5. Aggregated Evidence (real facts/links gathered by tools and/or documents, if any)
                    6. Research Findings (papers and key findings, if the query was complex enough to trigger research)

                    Generate EXACTLY {approach_count} unique candidate approaches.

                    Ground your approaches in the Aggregated Evidence and Research Findings where
                    relevant - do not ignore real evidence that was gathered for this query.

                    For each approach provide:
                    - approach_id
                    - title
                    - description
                    - reasoning_type
                    - advantages
                    - disadvantages
                    - estimated_confidence

                    Return ONLY valid JSON in exactly this format:

                    {{
                        "candidate_approaches": [
                            {{
                                "approach_id": 1,
                                "title": "",
                                "description": "",
                                "reasoning_type": "",
                                "advantages": [],
                                "disadvantages": [],
                                "estimated_confidence": 0
                            }}
                        ]
                    }}
                    """

    human_prompt = f"""
User Query:
{state['user_query']}

Query Analysis:
{json.dumps(state['query_analysis'], indent=2)}

Source Selection:
{json.dumps(state['source_selection'], indent=2)}

Reasoning & Evidence Plan:
{json.dumps(plan, indent=2)}

Aggregated Evidence:
{json.dumps(state.get('evidence', {}), indent=2)}

Research Findings:
{json.dumps(state.get('research_findings', {}), indent=2)}
"""

    response = llm.invoke([
        SystemMessage(content=prompt),
        HumanMessage(content=human_prompt)
    ])

    data = parse_json_response(response)

    if isinstance(data, list):
        candidate_approaches = data
    elif isinstance(data, dict) and "candidate_approaches" in data:
        candidate_approaches = data["candidate_approaches"]
    else:
        raise ValueError(f"Unexpected candidate_approaches response shape: {data}")

    state["candidate_approaches"] = candidate_approaches

    return state


def evaluate_candidate_approaches(state: State) -> State:

    system_prompt = """
                        You are the Evaluation Engine of an Explainable AI system.

                        Your responsibility is to evaluate each candidate approach objectively.

                        You will receive:
                        1. User Query
                        2. Query Analysis
                        3. Source Selection
                        4. Candidate Approaches

                        Evaluate EVERY approach independently.

                        Scoring Criteria:

                        1. Feasibility Score (0-100)
                        - How practical and implementable is this approach?

                        2. Relevance Score (0-100)
                        - How well does this approach address the user's intent?

                        3. Completeness Score (0-100)
                        - How completely does this approach solve the user's query?

                        4. Source Support Score (0-100)
                        - How well is this approach supported by the selected sources?

                        5. Overall Score (0-100)
                        - Overall quality considering all evaluation factors.

                        For each approach provide:

                        - approach_id
                        - title
                        - feasibility_score
                        - relevance_score
                        - completeness_score
                        - source_support_score
                        - strengths (list)
                        - weaknesses (list)
                        - assumptions (list)
                        - limitations (list)
                        - overall_score
                        - evaluation_summary

                        Rules:
                        - Evaluate ALL approaches.
                        - Be objective.
                        - Do NOT generate the final answer.
                        - Do NOT select the best approach.
                        - Return ONLY valid JSON.

                        Return exactly this format:

                        {
                            "evaluations":[
                                {
                                    "approach_id":1,
                                    "title":"",
                                    "feasibility_score":0,
                                    "relevance_score":0,
                                    "completeness_score":0,
                                    "source_support_score":0,
                                    "strengths":[],
                                    "weaknesses":[],
                                    "assumptions":[],
                                    "limitations":[],
                                    "overall_score":0,
                                    "evaluation_summary":""
                                }
                            ]
                        }
                        """

    human_prompt = f"""
User Query:
{state["user_query"]}

Query Analysis:
{json.dumps(state["query_analysis"], indent=2)}

Source Selection:
{json.dumps(state["source_selection"], indent=2)}

Candidate Approaches:
{json.dumps(state["candidate_approaches"], indent=2)}
"""

    response = llm.invoke([
        SystemMessage(content=system_prompt),
        HumanMessage(content=human_prompt)
    ])

    try:
        result = parse_json_response(response)
    except json.JSONDecodeError:
        raise ValueError("LLM returned invalid JSON.")

    if "evaluations" not in result:
        raise ValueError("Missing 'evaluations' in LLM response.")

    state["evaluations"] = result["evaluations"]

    return state


def select_best_approach(state: State):
    SELECT_BEST_PROMPT = """
                    You are the Decision Selection Engine.

                    Your task is to select the best candidate approach based on the evaluation results.

                    Input:
                    1. User Query
                    2. Query Analysis
                    3. Candidate Approaches
                    4. Evaluation Results

                    Instructions:
                    - Compare all candidate approaches.
                    - Select the single best approach.
                    - Explain why it was selected.
                    - Explain why the remaining approaches were not selected.
                    - Do NOT answer the user's query.
                    - Do NOT reveal chain of thought.
                    - Return ONLY valid JSON.

                    Output Format:

                    {
                    "selected_approach": {
                        "approach_id": 0,
                        "title": "",
                        "selection_reason": "",
                        "overall_confidence": 0
                    },
                    "rejected_approaches": [
                        {
                        "approach_id": 0,
                        "reason_for_rejection": ""
                        }
                    ]
                    }
                    """
    human_prompt = f"""
User Query:
{state["user_query"]}

Query Analysis:
{json.dumps(state["query_analysis"], indent=2)}

Candidate Approaches:
{json.dumps(state["candidate_approaches"], indent=2)}

Evaluations:
{json.dumps(state["evaluations"], indent=2)}
"""

    response = llm.invoke([
        SystemMessage(content=SELECT_BEST_PROMPT),
        HumanMessage(content=human_prompt)
    ])

    result = parse_json_response(response)

    if "selected_approach" not in result:
        raise ValueError("Missing 'selected_approach' in LLM response.")
    if "rejected_approaches" not in result:
        raise ValueError("Missing 'rejected_approaches' in LLM response.")

    state["selected_approach"] = result["selected_approach"]
    state["rejected_approaches"] = result["rejected_approaches"]

    return state


def generate_final_answer(state: State) -> State:

    ANSWER_GENERATION_PROMPT = """
                You are an expert AI assistant.

                You will receive:
                1. User Query
                2. Query Analysis
                3. Source Selection
                4. Selected Approach
                5. Evaluation Results
                6. Aggregated Evidence (real, live facts gathered from external
                   tools and/or an uploaded document, each with a source
                   URL if one exists)
                7. Research Findings (if this query triggered the Research Agent)

                Your task is to generate the final answer using ONLY the selected approach.

                Instructions:

                - Answer the user's query accurately and clearly.
                - Use the selected approach as the reasoning strategy.
                - Consider the selected sources when forming the answer.
                - If Aggregated Evidence or Research Findings contain relevant facts, USE
                  them and cite the source inline, e.g. "(Source: <name>, <url>)". Only
                  cite a URL that actually appears in Aggregated Evidence or Research
                  Findings - never invent one.
                - If both are empty or irrelevant, answer from your own knowledge and do
                  not fabricate a citation.
                - Produce a concise but complete response.
                - Do NOT mention internal reasoning.
                - Do NOT mention chain of thought.
                - Do NOT mention discarded approaches.
                - Do NOT output explanations or JSON other than the required format.

                Return ONLY valid JSON.

                Format:

                {
                    "answer":""
                }
                """

    human_prompt = f"""
User Query:
{state["user_query"]}

Query Analysis:
{json.dumps(state["query_analysis"], indent=2)}

Source Selection:
{json.dumps(state["source_selection"], indent=2)}

Selected Approach:
{json.dumps(state["selected_approach"], indent=2)}

Evaluation:
{json.dumps(state["evaluations"], indent=2)}

Aggregated Evidence:
{json.dumps(state.get("evidence", {}), indent=2)}

Research Findings:
{json.dumps(state.get("research_findings", {}), indent=2)}
"""

    response = llm.invoke([
        SystemMessage(content=ANSWER_GENERATION_PROMPT),
        HumanMessage(content=human_prompt)
    ])

    result = parse_json_response(response)

    if "answer" not in result:
        raise ValueError("Missing 'answer' in LLM response.")

    state["answer"] = result["answer"]

    return state


def generate_explainability(state: State) -> State:

    EXPLAINABILITY_PROMPT = """
You are the Explainability Engine of an Explainable AI system.

You are given:
1. User Query
2. Query Analysis
3. Source Selection
4. Candidate Approach Evaluations
5. Selected Approach
6. Final Answer
7. Reasoning & Evidence Plan (the strategy chosen before gathering evidence)
8. Aggregated Evidence (real data + links actually used, if any)
9. Research Findings (if the Research Agent ran for this query)

Your task is to generate a human-understandable explanation of how the final answer was produced.

IMPORTANT:
- Do NOT reveal internal chain of thought.
- Do NOT fabricate reasoning.
- Explain the decision process based ONLY on the supplied information.
- Explain why the selected approach was chosen.
- Explain why the other approaches were not selected.
- Mention the role of the selected sources and the reasoning strategy used.
- Mention whether the Research Agent ran, and why or why not.
- In "sources_used", list any external evidence actually used, including its
  name and URL/link exactly as it appears in Aggregated Evidence. If no
  external evidence was used, this can be an empty list.
- Keep explanations concise and user friendly.

Return ONLY valid JSON.

{
    "why_this_answer":"",
    "why_selected":"",
    "why_other_approaches_not_selected":[],
    "sources_used":[],
    "key_decision_factors":[],
    "confidence_score":0,
    "confidence_reason":"",
    "assumptions":[],
    "uncertainties":[],
    "limitations":[],
    "risk_level":"",
    "verification_suggestion":"",
    "follow_up_questions":[],
    "decision_flow":[]
}
"""

    human_prompt = f"""
User Query:
{state["user_query"]}

Query Analysis:
{json.dumps(state["query_analysis"], indent=2)}

Source Selection:
{json.dumps(state["source_selection"], indent=2)}

Approach Evaluations:
{json.dumps(state["evaluations"], indent=2)}

Selected Approach:
{json.dumps(state["selected_approach"], indent=2)}

Final Answer:
{state["answer"]}

Reasoning & Evidence Plan:
{json.dumps(state.get("reasoning_plan", {}), indent=2)}

Aggregated Evidence:
{json.dumps(state.get("evidence", {}), indent=2)}

Research Findings:
{json.dumps(state.get("research_findings", {}), indent=2)}
"""

    response = llm.invoke([
        SystemMessage(content=EXPLAINABILITY_PROMPT),
        HumanMessage(content=human_prompt)
    ])

    result = parse_json_response(response)

    state["explainability"] = result

    return state


def generate_summary(state: State) -> State:

    SUMMARY_PROMPT = """
You are the Summary Engine.

Generate a concise summary of the AI decision.

Include:

- Final Answer
- Selected Approach
- Why it was selected
- Confidence
- Sources Used
- Key Decision Factors
- Limitations
- Risk Level
- Verification Suggestion

Return ONLY valid JSON.

Format:

{
    "summary":{
        "selected_approach":"",
        "why_selected":"",
        "confidence":0,
        "sources_used":[],
        "key_decision_factors":[],
        "limitations":[],
        "risk_level":"",
        "verification_suggestion":"",
        "final_answer":""
    }
}
"""

    human_prompt = f"""
Answer:
{state["answer"]}

Selected Approach:
{json.dumps(state["selected_approach"], indent=2)}

Explainability:
{json.dumps(state["explainability"], indent=2)}

Source Selection:
{json.dumps(state["source_selection"], indent=2)}
"""

    response = llm.invoke([
        SystemMessage(content=SUMMARY_PROMPT),
        HumanMessage(content=human_prompt)
    ])

    state["summary"] = parse_json_response(response)

    return state


def generate_reasoning_tree(state: State) -> State:

    REASONING_TREE_PROMPT = """
You are the Reasoning Tree Generator.

You are given:
1. User Query
2. Candidate Approaches
3. Evaluation Results
4. Selected Approach
5. Explainability

Your task is to generate an interactive reasoning tree.

Each candidate approach becomes one node.

For every node provide:

- id
- title
- confidence
- selected (true/false)
- parent
- why
- strengths
- weaknesses
- assumptions
- limitations

Also generate one Final Answer node connected only to the selected approach.

Return ONLY valid JSON.

Format:

{
    "root":{
        "id":"query",
        "label":"User Query"
    },

    "nodes":[
        {
            "id":"",
            "parent":"",
            "title":"",
            "confidence":0,
            "selected":false,
            "why":"",
            "strengths":[],
            "weaknesses":[],
            "assumptions":[],
            "limitations":[]
        }
    ],

    "final_node":{
        "id":"final",
        "parent":"",
        "label":"Final Answer"
    }
}
"""

    human_prompt = f"""
User Query:
{state["user_query"]}

Candidate Approaches:
{json.dumps(state["candidate_approaches"], indent=2)}

Evaluations:
{json.dumps(state["evaluations"], indent=2)}

Selected Approach:
{json.dumps(state["selected_approach"], indent=2)}

Explainability:
{json.dumps(state["explainability"], indent=2)}
"""

    response = llm.invoke([
        SystemMessage(content=REASONING_TREE_PROMPT),
        HumanMessage(content=human_prompt)
    ])

    state["reasoning_tree"] = parse_json_response(response)

    return state


## Graph — original edges kept, 2 new conditional branch points added

**Branch point 1** (after `source_selection`): routes to `rag_node` if a document was uploaded this turn, and/or to `tool_execution` if `reasoning_plan.query_complexity` is not `"Simple"`. A `Simple` query with no document uploaded skips both and goes straight to `evidence_aggregation` — this is the "don't search multiple tools on a normal query" behavior you asked for.

**Branch point 2** (after `evidence_aggregation`): routes to `research_node` only if `query_complexity == "Complex"`, otherwise straight to `candidate_generation`.

Everything else — the 13 original nodes and their edges — is unchanged.

In [ ]:
def route_after_source_selection(state: State) -> List[str]:
    branches = []

    if state.get("document_uploaded"):
        branches.append("rag_node")

    complexity = (state.get("reasoning_plan", {}) or {}).get("query_complexity", "Moderate")
    if complexity != "Simple":
        branches.append("tool_execution")

    return branches or ["evidence_aggregation"]


def route_after_evidence_aggregation(state: State) -> str:
    complexity = (state.get("reasoning_plan", {}) or {}).get("query_complexity", "Moderate")
    if complexity == "Complex":
        return "research_node"
    return "candidate_generation"


In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.sqlite import SqliteSaver
import sqlite3

builder = StateGraph(State)

# Original nodes
builder.add_node("query_analysis", query_analysis)
builder.add_node("missing_context_detector", missing_context_detector)
builder.add_node("reasoning_evidence_planner", reasoning_evidence_planner)
builder.add_node("source_selection", source_selection)
builder.add_node("tool_execution", tool_execution)
builder.add_node("evidence_aggregation", evidence_aggregation)
builder.add_node("candidate_generation", generate_candidate_approaches)
builder.add_node("evaluation", evaluate_candidate_approaches)
builder.add_node("select_best", select_best_approach)
builder.add_node("final_answer", generate_final_answer)
builder.add_node("explainability", generate_explainability)
builder.add_node("reasoning_tree", generate_reasoning_tree)
builder.add_node("summary", generate_summary)

# New nodes
builder.add_node("rag_node", rag_node)
builder.add_node("research_node", research_node)

# Original linear edges, unchanged up through source_selection
builder.add_edge(START, "query_analysis")
builder.add_edge("query_analysis", "missing_context_detector")
builder.add_edge("missing_context_detector", "reasoning_evidence_planner")
builder.add_edge("reasoning_evidence_planner", "source_selection")

# NEW: conditional branch instead of the old straight source_selection -> tool_execution edge
builder.add_conditional_edges(
    "source_selection",
    route_after_source_selection,
    ["rag_node", "tool_execution", "evidence_aggregation"],
)
builder.add_edge("rag_node", "evidence_aggregation")
builder.add_edge("tool_execution", "evidence_aggregation")

# NEW: conditional branch instead of the old straight evidence_aggregation -> candidate_generation edge
builder.add_conditional_edges(
    "evidence_aggregation",
    route_after_evidence_aggregation,
    ["research_node", "candidate_generation"],
)
builder.add_edge("research_node", "candidate_generation")

# Original edges, unchanged from here on
builder.add_edge("candidate_generation", "evaluation")
builder.add_edge("evaluation", "select_best")
builder.add_edge("select_best", "final_answer")
builder.add_edge("final_answer", "explainability")
builder.add_edge("explainability", "reasoning_tree")
builder.add_edge("reasoning_tree", "summary")
builder.add_edge("summary", END)

conn = sqlite3.connect("chatbot.db", check_same_thread=False)
checkpointer = SqliteSaver(conn=conn)

graph = builder.compile(checkpointer=checkpointer)
graph


## Test runs

One simple query (no document, expect `tool_execution`/`rag_node`/`research_node` all skipped), one with a document uploaded, one complex query.

In [ ]:
# Simple query, no document -> should skip tool_execution, rag_node, and research_node entirely
config = {"configurable": {"thread_id": "test-simple"}}
result = graph.invoke({
    "user_query": "Hello, what can you help me with?",
    "document_uploaded": False,
    "document_path": None,
}, config=config)
print("SIMPLE complexity seen:", result["reasoning_plan"].get("query_complexity"))
print("ANSWER:", result["answer"])


In [ ]:
# Document uploaded -> should trigger rag_node regardless of complexity
config = {"configurable": {"thread_id": "test-document"}}
result = graph.invoke({
    "user_query": "What tools were used to build this system?",
    "document_uploaded": True,
    "document_path": "EduRAG_Manuscript.pdf",
}, config=config)
print("PDF chunks retrieved:", len(result.get("pdf_context", [])))
print("ANSWER:", result["answer"])


In [ ]:
# Complex query -> should trigger both tool_execution and research_node
config = {"configurable": {"thread_id": "test-complex"}}
result = graph.invoke({
    "user_query": "Compare the latest research on transformer efficiency techniques and explain which is most promising, citing sources.",
    "document_uploaded": False,
    "document_path": None,
}, config=config)
print("Complexity seen:", result["reasoning_plan"].get("query_complexity"))
print("Research findings:", json.dumps(result.get("research_findings", {}), indent=2)[:500])
print("ANSWER:", result["answer"])
